# Logos Pretraining - Colab GPU

End-to-end pretraining of a single **Logos** transformer using the current KDA/SWA/CSA/HCA architecture.

**Architecture**
- KDA recurrent/chunkwise attention.
- SWA local sliding-window attention.
- CSA learned 4-token compression plus sparse top-k global recall.
- HCA learned heavier compression plus dense compressed global recall.
- Entry / Body / Exit looped-depth structure with Block Attention Residuals.
- MoE FFNs with optional cross-loop expert diversity.


In [ ]:
!nvidia-smi


In [ ]:
import pathlib, subprocess
REPO_URL = 'https://github.com/Rorical/Logos.git'
REPO_PATH = pathlib.Path('/content/Logos')
if REPO_PATH.exists():
    subprocess.check_call(['git', '-C', str(REPO_PATH), 'fetch', '--all', '--prune'])
    subprocess.check_call(['git', '-C', str(REPO_PATH), 'reset', '--hard', 'origin/master'])
else:
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_PATH)])
%cd /content/Logos
!pip install -q -U uv
!uv sync --extra wandb


In [ ]:
import sys, torch
sys.path.insert(0, '/content/Logos')
from scripts.train import build_arg_parser, main
assert torch.cuda.is_available(), 'Switch this Colab runtime to a GPU.'
print('torch:', torch.__version__)
print('device:', torch.cuda.get_device_name(0))


In [ ]:
parser = build_arg_parser()
cli = [
    '--model', 'logos',
    '--streaming',
    '--dataset', 'HuggingFaceFW/fineweb-edu',
    '--dataset-config', 'sample-10BT',
    '--text-column', 'text',
    '--val-docs', '200',
    '--tiktoken-encoding', 'cl100k_base',
    '--total-tokens', '1B',
    '--batch-size', '2',
    '--max-length', '2048',
    '--log-every', '50',
    '--eval-every', '10000',
    '--save-every', '5000',
    '--sample-every', '20000',
    '--keep-last-n', '5',
    '--num-workers', '8',
    '--prefetch-factor', '8',
    '--bf16',
    '--compile',
    '--compile-mode', 'max-autotune',
    '--gradient-checkpointing',
    '--ckpt-granularity', 'per-block',
    '--block-residual-isolate-softmax',
    '--diagnostic-every', '100',
    '--d-model', '768',
    '--num-heads', '12',
    '--head-dim', '64',
    '--d-ff', '2048',
    '--num-entry-layers', '2',
    '--num-body-layers', '4',
    '--num-exit-layers', '2',
    '--num-loops', '3',
    '--chunk-size', '128',
    '--conv-size', '4',
    '--entry-attn-pattern', 'hca,kda',
    '--body-attn-pattern', 'hca,csa,kda,swa,csa,csa,kda,swa,csa,hca,kda,swa',
    '--exit-attn-pattern', 'csa,swa',
    '--csa-compression', '4',
    '--csa-top-k', '32',
    '--csa-indexer-heads', '4',
    '--csa-indexer-dim', '32',
    '--hca-compression', '128',
    '--swa-window', '256',
    '--use-moe',
    '--num-shared-experts', '2',
    '--num-sparse-experts', '32',
    '--top-k', '4',
    '--entry-top-k', '8',
    '--exit-top-k', '8',
    '--expert-d-ff', '512',
    '--moe-diversity-factor', '0.5',
    '--bias-update-rate', '0.02',
    '--moe-log-every', '1000',
    '--lm-head-chunk-size', '1024',
    '--muon',
    '--muon-schedule-hyperparams',
    '--lr', '0.004',
    '--embed-lr', '0.1',
    '--muon-lr', '0.01',
    '--router-lr', '1e-3',
    '--router-warmup-steps', '1000',
    '--scheduler', 'wsd',
    '--warmup-steps', '1000',
    '--decay-steps', '20000',
    '--opt-state-log-every', '1000',
    '--ema-decay', '0.999',
    '--sample-prompt', 'In a recent study, researchers found that',
    '--sample-max-tokens', '80',
    '--save-dir', '/content/logos-pretraining',
    '--seed', '42',
    '--wandb',
    '--wandb-project', 'logos-pretrain',
    '--wandb-mode', 'offline',
]
args = parser.parse_args(cli)
for k, v in sorted(vars(args).items()):
    print(f'{k:<28} {v}')


In [ ]:
main(args)
